In [ ]:
# Before starting this notebook, create a Google Earth Engine account and set up a cloud project
#Load Google Earth Engine and geemap Python libraries (you may have to install these before)

import ee
import geemap
import geopandas
import numpy as np
import matplotlib as plt

# Trigger the authentication flow.
ee.Authenticate()

#Change varibles with GEE project ID, desired dates, bands, and output file paths
gee_proj_id= 'project-ID'
start_date='2025-11-01'
end_date='2025-11-30'
band_selection= 'sm_rootzone'
#add _mean to the the original band name
mean_band_selection='sm_rootzone_mean'
output_stats = " /path.csv" 
output_raster= "/path.tif"

# Initialize the project with your cloud project ID
ee.Initialize(project=gee_proj_id)



In [ ]:
# Function to calculate zonal statistics and export to a file
######################TODO - figure out how to print statistic value ######################

# Define output path 
#out_stats = "/Users/amgi2571/Desktop/Spring 2026/remote sensing envi/final_proj/smap_zonal_stats_2016.csv" 

def get_mean(raster, zones, output_file):
    """Input raster, zone (or study region), output file path, statistic type, and scale. Allowed output formats: csv, shp, json, kml, kmz. Allowed statistics type: MEAN, MAXIMUM, MINIMUM, MEDIAN, STD, MIN_MAX, VARIANCE, SUM """    
    zone_stats = geemap.zonal_stats(raster, zones, output_file, stat_type="MEAN", scale=1000)
    return print(zone_stats)


In [ ]:
# load smap data from GEE and explore available bands 

smap_collection = (
    ee.ImageCollection('NASA/SMAP/SPL4SMGP/008')
    .filterDate(start_date, end_date)
)
smap_collection

# Select one band to focus on
smap_bands = smap_collection.select([band_selection])

# Apply Mean Reducer - this calculates the mean raster of your selected band
mean_smap_raster = smap_bands.reduce(ee.Reducer.mean())


In [ ]:
# Load the states you would like to clip your data to

states = ee.FeatureCollection("TIGER/2018/States")
lower48 = states.filter(ee.Filter.inList('NAME', [
    'Alabama', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut', 
    'Delaware', 'Florida', 'Georgia', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 
    'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 
    'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 
    'Nevada', 'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 
   'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 
    'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 
    'Utah', 'Vermont', 'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 
    'Wyoming'
]))

country_geometry = lower48.union()

In [ ]:
#calculate mean of all band pixel values 
#Output data is a single value representing the mean rootzone soil moisture of all pixels in the monthly mean raster calculated above

get_mean(mean_smap_raster, country_geometry, output_stats)


In [ ]:
# Create a visualization of sm_rootzone in the lower 48 states

m = geemap.Map(center=[40, -100], zoom=4)
clipped_image =  mean_smap_raster.clip(country_geometry)

#select the SMAP band to visualize and color palette
vis_params = {
    'bands': [mean_band_selection],
    'min': 0,
    'max': 0.9,
    'palette': 'Spectral',
}
m.add_layer(clipped_image, vis_params, 'Mean SM')
m

In [ ]:
#Download the mean raster to your computer (must clip to bounding box - working on this to clip to image collection)
######################TODO - figure out how to clip to lower48 ######################

roi = ee.Geometry.Rectangle([-124.848974, 24.396308, -66.885444, 49.384358])
geemap.ee_export_image(clipped_image, filename=output_raster, scale=2000, region=roi)